## Load Wandb database to csv

In [ ]:
import pickle
import pandas as pd
import wandb
from omegaconf import OmegaConf

In [3]:
api = wandb.Api()

# Project is specified by <entity/project-name>
runs = api.runs("Long_Memory_Arena/Long_Memory_Arena_LF_GRU")

summary_list, config_list, name_list, tag_list = [], [], [], []

for i, run in enumerate(runs):
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)

    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})

    # .name is the human-readable name of the run.
    name_list.append(run.name)

    tag_list.append(run.tags)


df = pd.DataFrame(
    {"summary": summary_list, "config": config_list, "name": name_list, "tag": tag_list}
)

with open("project_data.pkl", "wb") as f:
    pickle.dump(df, f)

# Data Processing

In [6]:
model_list = set(["tcn", "gru", "mamba", "s4", "rnn", "lstm"])


def convert_tags_to_string_if_target_present(tags, target_tags):
    # Check if any of the target tags is in the tag list
    for tag in tags:
        if tag.lower() in target_tags:
            return tag.lower()


with open("project_data.pkl", "rb") as f:
    df = pickle.load(f)

rho_name = "impulse"
df["data.rho_name"] = df.config.apply(lambda x: x.get("data"))
df = df.dropna()
df["data.rho_name"] = df.config.apply(lambda x: x["data"]["rho_name"])
df["data.rho_param"] = df.config.apply(lambda x: x["data"]["rho_param"])
df["tags"] = df.config.apply(lambda x: x["run_specs"]["tags"])
df["param_total"] = df.config.apply(lambda x: x["model/params/total"])

df["val/loss"] = df["summary"].apply(lambda x: x.get("val/loss", None))
df["train/rLoss1_epoch.min"] = df["summary"].apply(lambda x: x.get("train/rLoss1_epoch.min", None))
df["time"] = df["summary"].apply(lambda x: x.get("_timestamp", None))

# Add a model name based on tags
df["model"] = df["tags"].apply(lambda x: convert_tags_to_string_if_target_present(x, model_list))

df = df[
    [
        "time",
        "model",
        "tags",
        "data.rho_name",
        "data.rho_param",
        "param_total",
        "train/rLoss1_epoch.min",
    ]
]
df.to_csv("project.csv")

In [7]:
df

,time,model,tags,data.rho_name,data.rho_param,param_total,train/rLoss1_epoch.min
0,1.736245e+09,gru,"[lf, gru, airy]",airy,{'alpha': 1},6433,0.398528
1,1.736245e+09,gru,"[lf, gru, airy]",airy,{'alpha': 0.8571428571428571},6433,0.445079
2,1.736245e+09,gru,"[lf, gru, airy]",airy,{'alpha': 0.7142857142857142},6433,0.403222
3,1.736245e+09,gru,"[lf, gru, airy]",airy,{'alpha': 0.5714285714285714},6433,0.277769
4,1.736242e+09,gru,"[lf, gru, airy]",airy,{'alpha': 0.42857142857142855},6433,0.453053
...,...,...,...,...,...,...,...
123,1.736211e+09,gru,"[lf, gru, impulse]",impulse,{'alpha': 0.2857142857142857},6433,1.000002
124,1.736210e+09,gru,"[lf, gru, impulse]",impulse,{'alpha': 0.42857142857142855},6433,1.000005
125,1.736214e+09,gru,"[lf, gru, impulse]",impulse,{'alpha': 0.7142857142857142},6433,0.999889
126,1.736210e+09,gru,"[lf, gru, impulse]",impulse,{'alpha': 0.2857142857142857},6433,1.000003
